# Proyección de Ventas Netas - Urvet México

## Metodología

El presente proyecto sigue una metodología estructurada de Ciencia de Datos aplicada a Series de Tiempo (Forecasting). El flujo de trabajo abarca desde la extracción y consolidación de registros históricos hasta el análisis descriptivo y la preparación de los datos (Feature Engineering) para los modelos predictivos (SARIMAX, Prophet, LSTM, KNN y Regresión Logística).

---

### Contexto general de la empresa

**Grupo Urvet** es una empresa mexicana familiar con más de 29 años de trayectoria, posicionada como uno de los distribuidores a nivel nacional más importantes en el sector especializado de la salud y nutrición animal. 

* **Portafolio principal:** Son distribuidores a nivel nacional de alimentos super premium (como *Diamond Pet Foods* y *Taste of the Wild*) y son representantes exclusivos del laboratorio **MSD** (enfocado en medicamentos e innovación para prevención y tratamiento de enfermedades en mascotas).
* **Operación y Alcance:** Cuentan con su Matriz en Guadalajara, Jalisco, y una extensa red de Centros de Distribución (CEDIS) en puntos clave de la República (CDMX, Monterrey, Querétaro, Tijuana, Culiacán, León, entre otros).
* **Mercado Objetivo:** Canal especializado que incluye clínicas veterinarias, hospitales veterinarios, pet shops y criaderos.

**El Problema de Negocio:**
Dada la naturaleza de su catálogo, una correcta planificación de inventario es vital para la rentabilidad de los CEDIS. El objetivo de este proyecto es **proyectar las ventas netas del mes de abril de 2026**, entendiendo la "Venta Neta" como el monto real ingresado tras la aplicación de descuentos comerciales y la deducción de devoluciones.

---

### Datos

Los datos provienen de los reportes internos de rentabilidad de Urvet, extraídos del sistema central de la empresa.

* **Estructura:** Se dispone de archivos mensuales en formato `.xlsx` y `.csv` que abarcan el histórico transaccional de los años 2024, 2025 y el primer trimestre de 2026.
* **Variable Objetivo (Target):** `subtotalmn` (Calculada a partir de los montos de facturación, descuentos y devoluciones).
* **Volumen:** Aproximadamente 700,000 registros transaccionales en su forma cruda.
* **Granulidad:** Cada uno de los registros indica una línea asociada a una factura, por lo que, pueden existir múltiples códigos de facturas `doc_id` para varios registros, pero estos registros relacionados deben en la fecha en custión, deben coincidir exactamente con el cliente `customer_id` y agente `route_id` que están involucrados,

---

## Recursos y configuraciones base

In [34]:
import sys
import os
from pathlib import Path
import glob
import openpyxl

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

print(f'Base directory: {root_path}')

import config.settings as config

import pandas as pd
import numpy as np

Base directory: /Users/angelvelasco/Library/CloudStorage/GoogleDrive-angel.emma.velasco@gmail.com/My Drive/Documentos/universidad/LIACD/S3/JAPI


### EDA (Análisis Exploratorio de Datos)

El objetivo de esta fase no es predecir, sino entender el comportamiento histórico de las ventas de Urvet, identificar patrones estacionales (ej. picos de venta en ciertas quincenas), limpiar anomalías y generar una "Fuente de Verdad" única que alimentará a los 4 especialistas en modelado del equipo.

#### 1. Recopilación

Como primer paso, consolidaremos todos los reportes mensuales aislados alojados en el directorio `data/` en un único *DataFrame* continuo para evaluar la serie de tiempo en su totalidad.

In [35]:
data_dir = Path(config.DATA_DIR) / "*.xlsx"
print(f"Data directory: '{data_dir}'")

files = glob.glob(str(data_dir))
files.sort()

if not files:
    print(f"No se encontraron archivos con el patrón '{data_dir}'")
else:
    print(f"Archivo encontrado: {len(files)}")

Data directory: '/Users/angelvelasco/Library/CloudStorage/GoogleDrive-angel.emma.velasco@gmail.com/My Drive/Documentos/universidad/LIACD/S3/JAPI/data/*.xlsx'
Archivo encontrado: 27


In [33]:
all_data = []

for file in files:
    try:
        if format == '.csv':
            df_temp = pd.read_csv(file)
        else:
            df_temp = pd.read_excel(file)

        if not df_temp.empty:
            df_temp['source_file'] = Path(file).name
            all_data.append(df_temp)
    except Exception as e:
        print(f"Error al leer {file}: {e}")

raw_data = pd.concat(all_data, ignore_index=True)
raw_data

,lugar,falta_fac,ctototmn,subtotalmn,ventacan,ventamon,devolucan,devolumon,utilidad,cant_surt,...,nom_cte,cve_factu,no_fac,doc_id,cur_id,nregavg,des_cse,des_lug,nom_age,source_file
0,GENERAL,2024-01-25,634.064240,1154.484000,1,1154.4840,0,0.0,520.419760,1,...,J REFUGIO ALVAREZ GUTIERREZ,GDL,409850,GDL 409850,VN,1,DIAMOND,ALMACEN GENERAL,Cesar Olmos Foraneo- R59 Vtas Gdl,Copia de rentabilidad_2024_01.xlsx
1,GENERAL,2024-01-25,67.586323,120.689700,2,120.6897,0,0.0,53.103377,2,...,J REFUGIO ALVAREZ GUTIERREZ,GDL,409852,GDL 409852,VN,1,VANGUARDIA ACUARISTICA,ALMACEN GENERAL,Cesar Olmos Foraneo- R59 Vtas Gdl,Copia de rentabilidad_2024_01.xlsx
2,GENERAL,2024-01-29,0.000000,-69.270000,0,0.0000,0,0.0,-69.270000,0,...,J REFUGIO ALVAREZ GUTIERREZ,GDL,409850,GDL 409850,VN,1,DIAMOND,ALMACEN GENERAL,Cesar Olmos Foraneo- R59 Vtas Gdl,Copia de rentabilidad_2024_01.xlsx
3,GENERAL,2024-01-29,0.000000,-7.240000,0,0.0000,0,0.0,-7.240000,0,...,J REFUGIO ALVAREZ GUTIERREZ,GDL,409852,GDL 409852,VN,1,VANGUARDIA ACUARISTICA,ALMACEN GENERAL,Cesar Olmos Foraneo- R59 Vtas Gdl,Copia de rentabilidad_2024_01.xlsx
4,GENERAL,2024-01-02,622.240000,1134.719395,1,1134.7194,0,0.0,512.479395,1,...,RIBERAS PETS STORE,GDL,407356,GDL 407356,VN,1,DIAMOND NATURALS,ALMACEN GENERAL,Cesar Olmos Foraneo- R59 Vtas Gdl,Copia de rentabilidad_2024_01.xlsx
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636328,CDMX,2026-03-05,196.380000,542.931028,1,542.9310,0,0.0,346.551028,1,...,DOGTOR PEEK,MX,69910,MX 69910,VN,1,TASTE OF THE WILD,CDMX- Vallejo,ABRAHAM ENRIQUEZ Benito Juarez R122 CDMX,Copia de rentabilidad_2026_03.xlsx
636329,PUEBLA,2026-03-05,369.633145,714.724150,1,714.7242,0,0.0,345.091005,1,...,MAURICIO DANIEL MONTERROSAS,PUE,5435,PUE 5435,VN,1,DIAMOND,PUEBLA,Julio Hernandez AG R143 PUEBLA,Copia de rentabilidad_2026_03.xlsx
636330,PUEBLA,2026-03-05,635.650127,1219.406871,1,1219.4069,0,0.0,583.756744,1,...,MAURICIO DANIEL MONTERROSAS,PUE,5435,PUE 5435,VN,1,DIAMOND,PUEBLA,Julio Hernandez AG R143 PUEBLA,Copia de rentabilidad_2026_03.xlsx
636331,PUEBLA,2026-03-05,324.455298,627.206871,1,627.2069,0,0.0,302.751573,1,...,MAURICIO DANIEL MONTERROSAS,PUE,5435,PUE 5435,VN,1,DIAMOND,PUEBLA,Julio Hernandez AG R143 PUEBLA,Copia de rentabilidad_2026_03.xlsx


In [38]:
raw_data.columns

Index(['lugar', 'falta_fac', 'ctototmn', 'subtotalmn', 'ventacan', 'ventamon',
       'devolucan', 'devolumon', 'utilidad', 'cant_surt', 'cse_prod',
       'cve_age', 'cve_cte', 'cve_prod', 'desc_prod', 'nom_cte', 'cve_factu',
       'no_fac', 'doc_id', 'cur_id', 'nregavg', 'des_cse', 'des_lug',
       'nom_age', 'source_file'],
      dtype='str')

In [36]:
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 636333 entries, 0 to 636332
Data columns (total 25 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   lugar        636333 non-null  str           
 1   falta_fac    636333 non-null  datetime64[us]
 2   ctototmn     636333 non-null  float64       
 3   subtotalmn   636333 non-null  float64       
 4   ventacan     636333 non-null  int64         
 5   ventamon     636333 non-null  float64       
 6   devolucan    636333 non-null  int64         
 7   devolumon    636333 non-null  float64       
 8   utilidad     636333 non-null  float64       
 9   cant_surt    636333 non-null  int64         
 10  cse_prod     635584 non-null  str           
 11  cve_age      636333 non-null  int64         
 12  cve_cte      636333 non-null  int64         
 13  cve_prod     635584 non-null  object        
 14  desc_prod    635584 non-null  str           
 15  nom_cte      636333 non-null  str           


#### Limpieza y transformación de los datos

En esta fase preparamos los datos transaccionales crudos para convertirlos en una serie de tiempo apta para modelado predictivo. Las acciones principales que llevaremos a cabo son:

1. **Estandarización de columnas:** Limpieza de nombres (minúsculas, sin espacios) para garantizar la consistencia entre los archivos de 2024, 2025 y 2026.
2. **Conversión de tipos de datos:** Asegurar que las fechas sean objetos `datetime` y las métricas financieras sean valores numéricos.
3. **Imputación de Nulos:** Tratamiento de valores faltantes en campos financieros (asumiendo que un nulo en un descuento o devolución equivale a cero).
4. **Agrupación Temporal (Resampling):** Consolidación de los ~700,000 registros transaccionales en un DataFrame de granularidad diaria, estructurando la serie temporal que consumirán los modelos estadísticos y de Deep Learning.

In [ ]:
# Mapa de columnas para renombrar
column_mapping = {
    'doc_id': 'doc_id', # id del documento relacionado
    'falta_fac': 'sale_date',
    'ctototmn': 'total_cost',
    'subtotalmn': 'net_price',
    'ventamon': 'gross_price',
    'utilidadmn': 'profit',
    'cant_surt': 'quantity',
    'cse_prod': 'product_class_id',
    'cve_prod': 'product_id',
    'cve_age': 'route_id',
    'cve_cte': 'customer_id',
}

##### Descripción de las variables

* **`doc_id` (ID del Documento):** Identificador único de la transacción (factura o remisión). Sirve para el control de integridad y eliminación de registros duplicados en el proceso de consolidación.
* **`sale_date` (Fecha de Venta):** Representa la fecha contable en la que se realizó la operación. Es la **columna vertebral** para los modelos de series de tiempo (SARIMAX, Prophet y LSTM), ya que dicta la estacionalidad diaria, semanal y mensual.
* **`total_cost` (Costo Total):** Valor en moneda nacional (MXN) de la mercancía a precio de costo. Es fundamental para el análisis de rentabilidad y la detección de márgenes inconsistentes.
* **`net_price` (Precio Neto - VARIABLE OBJETIVO):** Es el monto facturado después de aplicar descuentos y bonificaciones, pero antes de impuestos. **Esta es la variable que vamos a proyectar para el mes de abril.**
* **`gross_price` (Precio Bruto):** Venta calculada a precio de lista por cantidad. La diferencia entre esta columna y `net_price` permite cuantificar el impacto de la política de descuentos de Urvet.
* **`profit` (Utilidad):** Resultado de restar el costo total al precio neto. Es un KPI de salud financiera que ayuda a los modelos de clasificación (KNN/LogReg) a identificar transacciones o clientes de alto valor.
* **`quantity` (Cantidad):** Número de unidades físicas (piezas, sacos o cajas) surtidas. Permite validar si el comportamiento de la venta es por volumen de demanda o por ajustes inflacionarios en los precios.
* **`product_class_id` (Clase de Producto):** Clasificación jerárquica (ej. Alimento Super Premium, Medicamento MSD). Clave para realizar proyecciones segmentadas por línea de negocio.
* **`product_id` (ID del Producto):** Código único del SKU. Permite rastrear el ciclo de vida de productos específicos y su peso dentro de la venta total.
* **`route_id` (ID de Ruta / Agente):** Identifica al agente de ventas o la zona logística. Es una variable exógena vital para entender el rendimiento por territorio y asignar metas en abril.
* **`customer_id` (ID del Cliente):** Identificador de la clínica, hospital o pet shop. Permite analizar la recurrencia y el comportamiento de compra a nivel micro.